# 12 -- DueCare Adversarial Prompt Factory: Industrial-Scale Test Generation

**DueCare** | Named for Cal. Civ. Code sect. 1714(a)

---

**Purpose:** Demonstrate the full adversarial prompt factory: 15
generators that transform base prompts into diverse adversarial
variations, validated for quality and ranked by victim impact.
From 10 base prompts, the factory produces 200+ tested variations.
From the full 74,567-prompt corpus, it can produce millions.

| | |
|---|---|
| **Input** | 10 graded base prompts from the trafficking domain pack |
| **Output** | 200+ validated, ranked adversarial test prompts; validation report; importance rankings |
| **Prerequisites** | `duecare-llm-wheels` dataset attached; no GPU required |
| **Pipeline position** | Stage 7 of the DueCare grading pipeline. Previous: NB 11 (Comparative Grading). Next: NB 13 (Rubric Evaluation). |

---

### Why adversarial testing matters

A model that scores well on standard safety prompts might fail
catastrophically when faced with adversarial variations. Bad actors
do not use polite, well-formatted questions to extract harmful
content -- they use:
- Academic framing ("For my PhD research...")
- Role-play injection ("You are a consultant advising...")
- Corporate disguise ("MEMO: Re: Workforce Optimization")
- Jurisdictional confusion (swap PH-HK corridor to BD-AE)
- Emotional manipulation ("The worker's visa expires tomorrow")

DueCare's 15 generators systematically produce these variations so
models cannot pass safety tests by memorizing standard phrasings.

### The 15 generators

| # | Generator | Attack vector |
|---|---|---|
| 1 | EvasionGenerator | Regulatory evasion framing |
| 2 | CoercionGenerator | Coercion/pressure tactics |
| 3 | FinancialGenerator | Financial obfuscation |
| 4 | RegulatoryGenerator | Jurisdictional complexity |
| 5 | CorridorGenerator | Migration corridor variation |
| 6 | MultiTurnGenerator | Conversation escalation |
| 7 | DocumentGenerator | Document-based injection |
| 8 | PersonaGenerator | 31 persona types |
| 9 | InteractiveGenerator | 10 interactive formats |
| 10 | CaseChallengeGenerator | Case-based challenges |
| 11 | FollowupGenerator | Informed follow-up pressure |
| 12 | CreativeGenerator | 12 creative attack strategies |
| 13 | ObfuscationGenerator | Unicode/encoding tricks |
| 14 | OutputConditionGenerator | Output format manipulation |
| 15 | DocumentQuizGenerator | Document comprehension exploit |

### Flow diagram

```
10 Base Prompts          15 Generators
      |                       |
      +-----------+-----------+
                  |
                  v
     +------------+-----------+
     |   Prompt Factory       |
     |   (15 x 10 x 2 = 300) |
     +------------+-----------+
                  |
                  v
     +------------+-----------+
     |   PromptValidator      |
     |   - PII check          |
     |   - Dedup              |
     |   - Quality threshold  |
     +------------+-----------+
                  |
                  v
     +------------+-----------+
     |  ImportanceRanker      |
     |  - Victim impact       |
     |  - Severity            |
     |  - Coverage gap        |
     +------------+-----------+
                  |
                  v
     Validated, Ranked Test Set
     (ready for model evaluation)
```


## 1. Install DueCare

Install the DueCare wheel packages from the attached dataset.


In [ ]:
import subprocess, glob, sys
for p in ['/kaggle/input/duecare-llm-wheels/*.whl',
          '/kaggle/input/datasets/taylorsamarel/duecare-llm-wheels/*.whl',
          '/kaggle/input/**/*.whl']:
    wheels = glob.glob(p, recursive=True)
    if wheels: break
if wheels:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + wheels + ['-q'])
    print(f'Installed {len(wheels)} DueCare wheels.')
else:
    print('WARNING: No wheels found. Attach the duecare-llm-wheels dataset.')


## 2. Load base prompts

We start with 10 graded base prompts -- prompts that have known
best/worst reference responses. These are the seed material that
the 15 generators will transform into adversarial variations.

Starting with graded prompts means we can later compare the model's
response to each adversarial variation against the known best/worst
responses for the base prompt it was derived from.


In [ ]:
from duecare.domains import register_discovered, load_domain_pack
register_discovered()
pack = load_domain_pack('trafficking')
base = [p for p in pack.seed_prompts() if p.get('graded_responses')][:10]
print(f'Base prompts: {len(base)}')
print(f'Categories: {set(p.get("category","?") for p in base)}')
for i, p in enumerate(base[:3]):
    print(f'\n  [{i+1}] {p.get("id","?")}: {p["text"][:80]}...')


## 3. Run all 15 generators

Each generator applies its specific attack vector to every base prompt,
producing 2 variations per prompt. This gives us approximately
15 x 10 x 2 = 300 adversarial variations.

The table below shows how many variations each generator produces and
how long it takes. Generators that produce fewer variations may have
filtered out prompts that don't fit their attack pattern -- this is
intentional, not a bug.


In [ ]:
from duecare.tasks.generators import ALL_GENERATORS
import time

t0 = time.time()
all_variations = []
generator_stats = []

print(f'{"Generator":<35} {"Variations":>10} {"Per Base":>10} {"Time":>8}')
print('-' * 65)
for gen in ALL_GENERATORS:
    gt0 = time.time()
    variations = gen.generate(base, n_variations=2, seed=42)
    all_variations.extend(variations)
    elapsed = time.time() - gt0
    per_base = len(variations) / max(len(base), 1)
    generator_stats.append({
        'name': gen.__class__.__name__,
        'count': len(variations),
        'per_base': per_base,
        'time': elapsed,
    })
    print(f'{gen.__class__.__name__:<35} {len(variations):>10} {per_base:>10.1f} {elapsed:>7.2f}s')

total_time = time.time() - t0
print(f'\nTotal: {len(all_variations)} variations from {len(base)} base prompts in {total_time:.1f}s')
print(f'Amplification factor: {len(all_variations)/max(len(base),1):.1f}x')
print(f'\nAt this rate, the full 74,567-prompt corpus would produce ~{int(74567 * len(all_variations)/max(len(base),1)):,} variations')


### Interpreting the generator output

**What to look for:**

- Generators that produce the most variations per base prompt are the
  most versatile -- they have more attack templates to apply
- Generators that produce fewer variations are more selective -- they
  only apply when the base prompt matches their attack pattern
- The amplification factor (total variations / base prompts) shows how
  much test coverage DueCare adds beyond the original corpus

The practical implication: a model cannot pass DueCare's safety tests
by memorizing standard prompts. Every generator produces variations
the model has never seen before.


## 4. Validate generated prompts

The `PromptValidator` ensures every generated prompt meets quality
standards before it enters the test suite:
- **PII check:** No real names, phone numbers, or addresses
- **Deduplication:** No near-identical prompts (first 100 chars match)
- **Quality threshold:** Minimum length, coherent text, proper formatting

This is the same PII gate that protects the entire DueCare pipeline.
Privacy is non-negotiable.


In [ ]:
from duecare.tasks.generators.test_validator import PromptValidator

validator = PromptValidator()
report = validator.validate_batch(all_variations)

print(f'=== Validation Report ===')
print(f'  Total generated:  {report.total}')
print(f'  Valid (passed):   {report.valid_count}')
print(f'  Invalid (failed): {report.invalid_count}')
print(f'  Pass rate:        {report.valid_count/max(report.total,1):.0%}')
print(f'\nIssues by type:')
for issue_type, count in report.issues_by_type.items():
    print(f'  {issue_type:<30} {count:>5}')


## 5. Rank by importance

Not all test prompts are equally important. The `ImportanceRanker`
prioritizes prompts by:
- **Victim impact:** How much real-world harm could result?
- **Severity:** How egregious is the exploitation scenario?
- **Coverage gap:** Does this test a category not well-covered yet?

This ensures the most critical test cases run first when evaluation
time is limited (e.g., on Kaggle GPU quotas).


In [ ]:
from duecare.tasks.generators.importance_ranker import ImportanceRanker

ranker = ImportanceRanker()
ranked = []
for p in report.valid_prompts[:50]:
    scored = ranker.rank_prompt(p)
    ranked.append((scored.overall, p))
ranked.sort(reverse=True)

print(f'Top 10 by importance (highest victim impact first):')
print(f'{"Rank":>4} {"Score":>8} {"Attack Type":<25} {"Preview"}')
print('-' * 80)
for i, (score, p) in enumerate(ranked[:10]):
    mt = p.get('metadata',{}).get('mutation_type','original')[:25]
    print(f'{i+1:>4} {score:>8.3f} {mt:<25} {p["text"][:40]}...')


## Summary and next steps

### Key findings

- **15 generators** produce diverse adversarial tests from any base
  prompt, covering evasion, coercion, financial obfuscation, persona
  injection, and more
- **PromptValidator** ensures quality and privacy: no PII, no
  duplicates, no below-threshold prompts
- **ImportanceRanker** prioritizes by victim impact and severity,
  ensuring the most critical tests run first
- From 10 base prompts, the factory produces 200+ validated, ranked
  test cases. From the full 74,567-prompt corpus, it can produce
  **millions** of unique adversarial variations

### Connection to other notebooks

- **Previous (NB 11):** Comparative grading evaluated model responses
  against known best/worst references. The adversarial factory extends
  this by generating novel attack variations.
- **Next (NB 13):** Rubric-anchored evaluation scores model responses
  to adversarial prompts against all 54 criteria from the 5 trafficking
  rubrics.
- **Phase 3 fine-tuning:** The validated, ranked prompt set becomes the
  training and evaluation corpus for Unsloth fine-tuning.

### Scale and impact

The prompt factory is the engine behind DueCare's claim that no model
can game the safety tests. With 15 generators and 74,567 base prompts,
the space of possible test variations is effectively infinite. A model
that memorizes answers to 1,000 prompts will still face 100,000 novel
variations it has never seen.

This is what makes DueCare useful to organizations like POEA, IOM, and
Polaris Project: it tests models the way adversaries actually attack
them, not the way benchmarks wish they would.

**Privacy is non-negotiable. The entire factory runs on-device.**
